# Дообучение stance-компонента Veriscope (QLoRA, Colab)

Цель: поднять качество классификации supports / refutes / not_enough_info, которая по калибровке на 75 клеймах даёт основную массу ошибок (вердикт «противоречиво» ошибался в 100% случаев из-за шумных stance-суждений).

План: Qwen2.5-7B-Instruct + QLoRA на FEVER (готовые пары claim+evidence из `copenlu/fever_gold_evidence`), промпт **в точности** как в `app/pipeline/stance.py` — модель учится в том формате, в котором её вызывает пайплайн.

Требования: Colab с GPU (T4 достаточно), ~2–4 часа на 30k примеров.

In [ ]:
%pip install -q unsloth datasets scikit-learn

## Данные: FEVER с готовым текстом доказательств

Метки FEVER переводим в термины пайплайна, классы балансируем.

In [ ]:
import json
import random

from datasets import load_dataset

LABEL_MAP = {
    "SUPPORTS": "supports",
    "REFUTES": "refutes",
    "NOT ENOUGH INFO": "not_enough_info",
}

STANCE_SYSTEM = (
    "You are a fact-checking assistant. Given a claim and an excerpt from a source, "
    "decide the stance of the source towards the claim. "
    'Respond with a JSON object: {"stance": "supports" | "refutes" | "not_enough_info", '
    '"rationale": "<one short sentence in the language of the claim>"}. '
    "Choose not_enough_info when the excerpt neither clearly confirms nor contradicts the claim."
)


def evidence_text(evidence) -> str:
    parts = []
    for item in evidence:
        if isinstance(item, (list, tuple)):
            strings = [str(part) for part in item if isinstance(part, str)]
            if strings:
                parts.append(max(strings, key=len))
        elif isinstance(item, str):
            parts.append(item)
    return " ".join(parts)[:1200]


def to_example(row) -> dict | None:
    label = LABEL_MAP.get(row["label"])
    excerpt = evidence_text(row["evidence"])
    if label is None or len(excerpt) < 40:
        return None
    user = f"CLAIM:\n{row['claim']}\n\nSOURCE TITLE:\n\n\nSOURCE EXCERPT:\n{excerpt}"
    target = json.dumps({"stance": label, "rationale": ""}, ensure_ascii=False)
    return {"system": STANCE_SYSTEM, "user": user, "target": target, "label": label}


raw = load_dataset("copenlu/fever_gold_evidence")
examples = [example for row in raw["train"] if (example := to_example(row))]
random.seed(42)
random.shuffle(examples)

PER_CLASS = 10000
buckets = {}
for example in examples:
    buckets.setdefault(example["label"], []).append(example)
train_rows = sum((bucket[:PER_CLASS] for bucket in buckets.values()), [])
random.shuffle(train_rows)

eval_rows = [example for row in raw["validation"] if (example := to_example(row))]
random.shuffle(eval_rows)
eval_rows = eval_rows[:600]

print(len(train_rows), "train /", len(eval_rows), "eval")
print({label: len(bucket) for label, bucket in buckets.items()})

## Модель

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    random_state=42,
)

## Бейзлайн до обучения (zero-shot macro-F1)

In [ ]:
import re

import torch
from sklearn.metrics import classification_report, f1_score

STANCE_VALUES = {"supports", "refutes", "not_enough_info"}


def predict(rows, batch_note=""):
    FastLanguageModel.for_inference(model)
    predictions = []
    for index, row in enumerate(rows):
        messages = [
            {"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            output = model.generate(inputs, max_new_tokens=60, do_sample=False)
        text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
        match = re.search(r'"stance"\s*:\s*"([a-z_]+)"', text)
        value = match.group(1) if match else ""
        predictions.append(value if value in STANCE_VALUES else "not_enough_info")
        if (index + 1) % 50 == 0:
            print(f"{batch_note}{index + 1}/{len(rows)}")
    return predictions


baseline_rows = eval_rows[:300]
gold = [row["label"] for row in baseline_rows]
baseline_pred = predict(baseline_rows, "baseline ")
print(classification_report(gold, baseline_pred, digits=3))
baseline_f1 = f1_score(gold, baseline_pred, average="macro")
print("zero-shot macro-F1:", round(baseline_f1, 3))

## Обучение

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer


def to_chat_text(row) -> dict:
    messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
        {"role": "assistant", "content": row["target"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


train_dataset = Dataset.from_list([to_chat_text(row) for row in train_rows])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=25,
        output_dir="stance_lora_out",
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        seed=42,
        max_seq_length=2048,
        packing=False,
    ),
)
trainer.train()

## Оценка после обучения

In [ ]:
tuned_pred = predict(baseline_rows, "tuned ")
print(classification_report(gold, tuned_pred, digits=3))
tuned_f1 = f1_score(gold, tuned_pred, average="macro")
print(f"macro-F1: zero-shot {baseline_f1:.3f} -> tuned {tuned_f1:.3f}")

## Экспорт: адаптер + GGUF для Ollama

In [ ]:
model.save_pretrained("stance_lora_adapter")
tokenizer.save_pretrained("stance_lora_adapter")
model.save_pretrained_gguf("stance_qwen_gguf", tokenizer, quantization_method="q4_k_m")

## Подключение к Veriscope

1. Скачай `stance_qwen_gguf/*.gguf` на свою машину.
2. Создай `Modelfile`:
```
FROM ./stance-qwen-q4_k_m.gguf
```
3. `ollama create qwen2.5-stance -f Modelfile`
4. В `.env` проекта: `LLM_MODEL=qwen2.5-stance`
5. Перегони калибровку и сравни таблицы до/после:
```
python -m scripts.calibrate data/calibration_full.jsonl
```